In [ ]:
# TAHAP 1 Instalasi Library Pendukung

!pip install -q geopandas libpysal esda splot mapclassify openpyxl joblib
print("[SUKSES] Library analisis berhasil disiapkan.")

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

from IPython.display import display
from libpysal.weights import Queen, KNN
from libpysal.weights.spatial_lag import lag_spatial
from esda.moran import Moran, Moran_Local

from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import (
    TimeSeriesSplit,
    RandomizedSearchCV
)

import joblib

warnings.filterwarnings("ignore")
np.random.seed(42)

# ----------------------------------------------------------------
# KONFIGURASI PENELITIAN
# ----------------------------------------------------------------
RANDOM_STATE = 42
K_UTAMA = 5
JUMLAH_LAG = 4
JUMLAH_FOLD = 5
TAHUN_TEST = 2025

path_folder = "/content/drive/MyDrive/skripsi"
path_output = os.path.join(path_folder, "output_analisis_revisi")
os.makedirs(path_output, exist_ok=True)

# True = analisis dihentikan jika pola pembagian rata tahunan terdeteksi.
HENTIKAN_JIKA_DISAGREGASI_RATA = False

print("Folder data  :", path_folder)
print("Folder output:", path_output)
print("KNN utama    :", f"k={K_UTAMA}")

# TAHAP 2 Penyelarasan & Disagregasi Data Excel (2022-2025)

# PENYELARASAN DATA BULANAN 2022–2025

# ================================================================
# 1. LOKASI FILE
# ================================================================

file_2022 = os.path.join(
    path_folder,
    "rekap_campak_2022.xlsx"
)

file_2023 = os.path.join(
    path_folder,
    "rekap_campak_2023.xlsx"
)

file_2024 = os.path.join(
    path_folder,
    "rekap_campak_2024.xlsx"
)

file_2025 = os.path.join(
    path_folder,
    "data_detail_campak_2025.xlsx"
)


# ================================================================
# 2. DAFTAR ACUAN 27 KECAMATAN KABUPATEN SUMENEP
# ================================================================

DAFTAR_KECAMATAN = [
    "AMBUNTEN",
    "ARJASA",
    "BATANG BATANG",
    "BATUAN",
    "BATUPUTIH",
    "BLUTO",
    "DASUK",
    "DUNGKEK",
    "GANDING",
    "GAPURA",
    "GAYAM",
    "GILIGENTENG",
    "GULUK GULUK",
    "KALIANGET",
    "KANGAYAN",
    "KOTA SUMENEP",
    "LENTENG",
    "MANDING",
    "MASALEMBU",
    "NONGGUNONG",
    "PASONGSONGAN",
    "PRAGAAN",
    "RAAS",
    "RUBARU",
    "SAPEKEN",
    "SARONGGI",
    "TALANGO"
]


# ================================================================
# 3. PEMETAAN VARIASI NAMA KECAMATAN
# ================================================================

MAPPING_KECAMATAN = {
    # Ambunten
    "AMBUTEN": "AMBUNTEN",

    # Arjasa
    "ARJASA PULAU KANGEAN": "ARJASA",
    "ARJASA (PULAU KANGEAN)": "ARJASA",
    "KEPULAUAN ARJASA": "ARJASA",

    # Batang Batang
    "BATANG-BATANG": "BATANG BATANG",
    "BATANGBATANG": "BATANG BATANG",

    # Batuputih
    "BATU PUTIH": "BATUPUTIH",
    "BATU-PUTIH": "BATUPUTIH",

    # Giligenteng
    "GILI GENTENG": "GILIGENTENG",
    "GILI-GENTENG": "GILIGENTENG",
    "GILI GENTING": "GILIGENTENG",
    "GILI-GENTING": "GILIGENTENG",
    "GILIGENTING": "GILIGENTENG",
    "GILI GENTENG PULAU GILI GENTING": "GILIGENTENG",
    "GILI GENTENG (PULAU GILI GENTING)": "GILIGENTENG",

    # Guluk Guluk
    "GULUK-GULUK": "GULUK GULUK",
    "GULUKGULUK": "GULUK GULUK",

    # Kota Sumenep
    "SUMENEP": "KOTA SUMENEP",
    "KECAMATAN KOTA SUMENEP": "KOTA SUMENEP",
    "KOTA": "KOTA SUMENEP",

    # Nonggunong
    "NONG GUNONG": "NONGGUNONG",
    "NONG-GUNONG": "NONGGUNONG"
}


# ================================================================
# 4. FUNGSI STANDARDISASI NAMA KECAMATAN
# ================================================================

def standardisasi_kecamatan(series):
    """
    Membersihkan dan menyeragamkan penulisan nama kecamatan.
    """
    hasil = series.astype("string")

    hasil = (
        hasil
        .str.strip()
        .str.upper()
        .str.replace(
            r"^(KECAMATAN|KEC\.?)\s+",
            "",
            regex=True
        )
        .str.replace(
            r"[_]+",
            " ",
            regex=True
        )
        .str.replace(
            r"\s*-\s*",
            "-",
            regex=True
        )
        .str.replace(
            r"\s+",
            " ",
            regex=True
        )
    )

    # Remove any text within parentheses (e.g., ' (PULAU SAPUDI)')
    hasil = hasil.str.replace(r"\s*\(.+\)", "", regex=True)
    hasil = hasil.str.strip()

    hasil = hasil.replace(MAPPING_KECAMATAN)

    return hasil


# ================================================================
# 5. FUNGSI NORMALISASI NAMA KOLOM
# ================================================================

def normalisasi_teks(teks):
    """
    Menyeragamkan teks agar pencarian nama kolom lebih fleksibel.
    """
    teks = str(teks).strip().upper()

    teks = re.sub(
        r"[^A-Z0-9]+",
        " ",
        teks
    )

    teks = re.sub(
        r"\s+",
        " ",
        teks
    ).strip()

    return teks


def cari_kolom(dataframe, kandidat, wajib=True):
    """
    Mencari nama kolom berdasarkan beberapa alternatif nama.
    """
    mapping_kolom = {
        normalisasi_teks(kolom): kolom
        for kolom in dataframe.columns
    }

    for nama in kandidat:
        nama_normal = normalisasi_teks(nama)

        if nama_normal in mapping_kolom:
            return mapping_kolom[nama_normal]

    if wajib:
        raise ValueError(
            "Kolom tidak ditemukan. "
            f"Kandidat kolom: {kandidat}. "
            f"Kolom tersedia: {list(dataframe.columns)}"
        )

    return None


# ================================================================
# 6. DAFTAR NAMA BULAN
# ================================================================

MAPPING_BULAN = {
    "1": 1, "01": 1, "JAN": 1, "JANUARI": 1, "JANUARY": 1,
    "2": 2, "02": 2, "FEB": 2, "FEBRUARI": 2, "FEBRUARY": 2,
    "3": 3, "03": 3, "MAR": 3, "MARET": 3, "MARCH": 3,
    "4": 4, "04": 4, "APR": 4, "APRIL": 4,
    "5": 5, "05": 5, "MEI": 5, "MAY": 5,
    "6": 6, "06": 6, "JUN": 6, "JUNI": 6, "JUNE": 6,
    "7": 7, "07": 7, "JUL": 7, "JULI": 7, "JULY": 7,
    "8": 8, "08": 8, "AGU": 8, "AGUSTUS": 8, "AUG": 8, "AUGUST": 8,
    "9": 9, "09": 9, "SEP": 9, "SEPTEMBER": 9,
    "10": 10, "OKT": 10, "OKTOBER": 10, "OCT": 10, "OCTOBER": 10,
    "11": 11, "NOV": 11, "NOVEMBER": 11,
    "12": 12, "DES": 12, "DESEMBER": 12, "DEC": 12, "DECEMBER": 12
}


def konversi_bulan(series):
    """
    Mengubah angka maupun nama bulan menjadi angka 1–12.
    """
    bulan_teks = (
        series
        .astype("string")
        .str.strip()
        .str.upper()
    )

    bulan_teks = bulan_teks.map(MAPPING_BULAN)

    return bulan_teks.astype("Int64")


# ================================================================
# 7. PEMERIKSAAN NAMA KECAMATAN
# ================================================================

def periksa_kecamatan(dataframe, nama_data):
    """
    Memeriksa nama wilayah yang belum sesuai dengan daftar acuan.
    """
    kecamatan_data = set(dataframe["Kecamatan"].dropna().unique())
    kecamatan_acuan = set(DAFTAR_KECAMATAN)

    tidak_dikenal = sorted(kecamatan_data - kecamatan_acuan)
    tidak_terdapat = sorted(kecamatan_acuan - kecamatan_data)

    print(f"\nPemeriksaan wilayah: {nama_data}")
    print("Jumlah kecamatan ditemukan:", len(kecamatan_data))

    if tidak_dikenal:
        print("Nama kecamatan belum dikenali:", tidak_dikenal)
        raise ValueError(
            f"Terdapat nama kecamatan yang belum distandardisasi pada {nama_data}: {tidak_dikenal}"
        )

    if tidak_terdapat:
        print("Kecamatan yang tidak muncul dalam data:", tidak_terdapat)
        print("Kecamatan tersebut akan diisi nol pada pembentukan balanced panel.")
    else:
        print("Seluruh 27 kecamatan ditemukan.")

def load_rekap_bulanan(file_path, tahun, allow_annual_distribution=False):
    """
    Membaca data historis bulanan.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File tidak ditemukan: {file_path}")

    df = pd.read_excel(file_path, engine="openpyxl")
    df.columns = [str(kolom).strip() for kolom in df.columns]

    print(f"\n[INFO] Membaca data {tahun}")
    print("Kolom tersedia:", list(df.columns))

    kolom_kecamatan = cari_kolom(
        df,
        kandidat=["Kecamatan", "Nama Kecamatan", "Kec"]
    )

    df = df.rename(columns={kolom_kecamatan: "Kecamatan"})
    df["Kecamatan"] = standardisasi_kecamatan(df["Kecamatan"])
    df = df.dropna(subset=["Kecamatan"]).copy()

    # ------------------------------------------------------------
    # Mencari bentuk LONG FORMAT
    # ------------------------------------------------------------
    kolom_bulan = cari_kolom(
        df,
        kandidat=["Bulan", "Month", "Bln"],
        wajib=False
    )

    kolom_jumlah = cari_kolom(
        df,
        kandidat=["Jumlah Kasus", "Jumlah_Kasus", "Total Kasus", "Kasus", "Jumlah"],
        wajib=False
    )

    df_long = pd.DataFrame()

    if kolom_bulan is not None and kolom_jumlah is not None:
        print("[INFO] Struktur data terdeteksi: LONG FORMAT")
        df_long = df[["Kecamatan", kolom_bulan, kolom_jumlah]].copy()
        df_long = df_long.rename(columns={
            kolom_bulan: "Bulan",
            kolom_jumlah: "Jumlah_Kasus"
        })
        df_long["Bulan"] = konversi_bulan(df_long["Bulan"])
        df_long["Jumlah_Kasus"] = pd.to_numeric(df_long["Jumlah_Kasus"], errors="coerce")

    # ------------------------------------------------------------
    # Mencari bentuk WIDE FORMAT atau ANNUAL TOTAL
    # ------------------------------------------------------------
    else:
        kolom_bulan_wide = {}
        for kolom in df.columns:
            teks_kolom = normalisasi_teks(kolom)
            teks_kolom = re.sub(r"\b20\d{2}\b", "", teks_kolom).strip()

            if teks_kolom in MAPPING_BULAN:
                nomor_bulan = MAPPING_BULAN[teks_kolom]
                kolom_bulan_wide[kolom] = nomor_bulan

        if len(kolom_bulan_wide) == 0:
            if kolom_jumlah is not None and allow_annual_distribution:
                print(
                    f"[INFO] Struktur data terdeteksi: TOTAL TAHUNAN. "
                    f"Mendistribusikan '{kolom_jumlah}' secara merata ke 12 bulan."
                )
                df_annual_total = df[[kolom_kecamatan, kolom_jumlah]].copy()
                df_annual_total = df_annual_total.rename(columns={
                    kolom_kecamatan: "Kecamatan",
                    kolom_jumlah: "Jumlah_Kasus"
                })

                list_of_dfs = []
                for month in range(1, 13):
                    temp_df = df_annual_total.copy()
                    temp_df["Bulan"] = month
                    temp_df["Jumlah_Kasus"] = temp_df["Jumlah_Kasus"] / 12
                    list_of_dfs.append(temp_df)
                df_long = pd.concat(list_of_dfs, ignore_index=True)
                df_long["Jumlah_Kasus"] = pd.to_numeric(df_long["Jumlah_Kasus"], errors="coerce")
            else:
                raise ValueError(
                    f"\nFile tahun {tahun} tidak mempunyai informasi bulanan.\n"
                    "File hanya terdeteksi sebagai data total tahunan.\n\n"
                    "Data yang dibutuhkan minimal berbentuk:\n"
                    "Kecamatan | Bulan | Jumlah Kasus\n\n"
                    "atau:\nKecamatan | Januari | Februari | ... | Desember"
                )
        else:
            print("[INFO] Struktur data terdeteksi: WIDE FORMAT")
            kolom_digunakan = (["Kecamatan"] + list(kolom_bulan_wide.keys()))
            df_wide = df[kolom_digunakan].copy()
            df_long = df_wide.melt(
                id_vars=["Kecamatan"],
                value_vars=list(kolom_bulan_wide.keys()),
                var_name="Nama_Bulan",
                value_name="Jumlah_Kasus"
            )
            df_long["Bulan"] = df_long["Nama_Bulan"].map(kolom_bulan_wide)
            df_long = df_long.drop(columns=["Nama_Bulan"])
            df_long["Jumlah_Kasus"] = pd.to_numeric(df_long["Jumlah_Kasus"], errors="coerce")

    # ------------------------------------------------------------
    # Pembersihan hasil
    # ------------------------------------------------------------
    df_long = df_long.dropna(subset=["Kecamatan", "Bulan", "Jumlah_Kasus"]).copy()
    df_long["Bulan"] = df_long["Bulan"].astype(int)
    df_long = df_long[df_long["Bulan"].between(1, 12)].copy()

    if (df_long["Jumlah_Kasus"] < 0).any():
        raise ValueError(f"Ditemukan jumlah kasus negatif pada data tahun {tahun}.")

    df_long["Tahun"] = tahun

    df_long = (
        df_long
        .groupby(["Kecamatan", "Tahun", "Bulan"], as_index=False)["Jumlah_Kasus"]
        .sum()
    )

    df_long["Jumlah_Kasus"] = df_long["Jumlah_Kasus"].round().astype(int)
    periksa_kecamatan(df_long, f"Data {tahun}")

    print(f"Jumlah kasus tahun {tahun}:", df_long["Jumlah_Kasus"].sum())
    print(f"Jumlah baris tahun {tahun}:", len(df_long))

    return df_long


# ================================================================
# 9. MEMBACA DATA HISTORIS BULANAN 2022–2024
# ================================================================

print("\n" + "=" * 75)
print("PROSES DATA HISTORIS BULANAN 2022–2024")
print("=" * 75)

df_2022_m = load_rekap_bulanan(file_2022, 2022, allow_annual_distribution=True)
df_2023_m = load_rekap_bulanan(file_2023, 2023, allow_annual_distribution=True)
df_2024_m = load_rekap_bulanan(file_2024, 2024, allow_annual_distribution=True)

df_historis = pd.concat([df_2022_m, df_2023_m, df_2024_m], ignore_index=True)

df_historis = (
    df_historis
    .sort_values(by=["Tahun", "Bulan", "Kecamatan"])
    .reset_index(drop=True)
)

print("\n=== RINGKASAN DATA HISTORIS BULANAN ===")
display(df_historis.head(15))
print("Total kasus historis:", df_historis["Jumlah_Kasus"].sum())
print("Jumlah baris historis:", len(df_historis))


# ================================================================
# 10. MEMBACA DAN MENGAGREGASI DATA DETAIL 2025
# ================================================================

print("\n" + "=" * 75)
print("PROSES DATA DETAIL TAHUN 2025")
print("=" * 75)

if not os.path.exists(file_2025):
    raise FileNotFoundError(f"File tidak ditemukan: {file_2025}")

df_2025_raw = pd.read_excel(file_2025, engine="openpyxl")
df_2025_raw.columns = [str(kolom).strip() for kolom in df_2025_raw.columns]

print("Kolom data 2025:", list(df_2025_raw.columns))

kolom_kecamatan_2025 = cari_kolom(
    df_2025_raw,
    kandidat=["Kecamatan", "Nama Kecamatan", "Kec"]
)

kolom_tanggal_2025 = cari_kolom(
    df_2025_raw,
    kandidat=["Tanggal Terima", "Tanggal Kejadian", "Tanggal Laporan", "Tanggal", "Date"]
)

df_2025_raw = df_2025_raw.rename(columns={
    kolom_kecamatan_2025: "Kecamatan",
    kolom_tanggal_2025: "Tanggal"
})

df_2025_raw["Kecamatan"] = standardisasi_kecamatan(df_2025_raw["Kecamatan"])
df_2025_raw["Tanggal"] = pd.to_datetime(df_2025_raw["Tanggal"], errors="coerce", dayfirst=True)

df_2025_clean = df_2025_raw.dropna(subset=["Kecamatan", "Tanggal"]).copy()
df_2025_clean = df_2025_clean[df_2025_clean["Tanggal"].dt.year == 2025].copy()
df_2025_clean = df_2025_clean[df_2025_clean["Kecamatan"] != "LUAR DAERAH"].copy()

if df_2025_clean.empty:
    raise ValueError("Tidak ada data valid dengan tanggal tahun 2025.")

df_2025_clean["Tahun"] = df_2025_clean["Tanggal"].dt.year
df_2025_clean["Bulan"] = df_2025_clean["Tanggal"].dt.month

periksa_kecamatan(df_2025_clean, "Data detail 2025")

df_2025_m = (
    df_2025_clean
    .groupby(["Kecamatan", "Tahun", "Bulan"], as_index=False)
    .size()
    .rename(columns={"size": "Jumlah_Kasus"})
)

df_2025_m["Jumlah_Kasus"] = df_2025_m["Jumlah_Kasus"].astype(int)
df_2025_m = df_2025_m.sort_values(by=["Tahun", "Bulan", "Kecamatan"]).reset_index(drop=True)
bulan_terakhir_2025 = int(df_2025_m["Bulan"].max())

print("\nBulan terakhir data 2025:", bulan_terakhir_2025)
print("Total kasus 2025:", df_2025_m["Jumlah_Kasus"].sum())
print("Jumlah baris agregasi 2025:", len(df_2025_m))
display(df_2025_m.head(15))


# ================================================================
# 11. MENGGABUNGKAN DATA 2022–2025
# ================================================================

df_master = pd.concat([df_historis, df_2025_m], ignore_index=True)
jumlah_duplikasi = df_master.duplicated(subset=["Kecamatan", "Tahun", "Bulan"]).sum()

print("\nJumlah kombinasi duplikat:", jumlah_duplikasi)

if jumlah_duplikasi > 0:
    raise ValueError("Masih terdapat duplikasi pada kombinasi Kecamatan, Tahun, dan Bulan.")


# ================================================================
# 12. MEMBENTUK BALANCED PANEL
# ================================================================

periode_awal = pd.Timestamp(year=2022, month=1, day=1)
periode_akhir = pd.Timestamp(year=2025, month=bulan_terakhir_2025, day=1)
daftar_periode = pd.date_range(start=periode_awal, end=periode_akhir, freq="MS")

grid = pd.MultiIndex.from_product(
    [DAFTAR_KECAMATAN, daftar_periode],
    names=["Kecamatan", "Periode"]
).to_frame(index=False)

grid["Tahun"] = grid["Periode"].dt.year
grid["Bulan"] = grid["Periode"].dt.month

df_final = grid.merge(df_master, on=["Kecamatan", "Tahun", "Bulan"], how="left")
df_final["Jumlah_Kasus"] = df_final["Jumlah_Kasus"].fillna(0).astype(int)

df_final = (
    df_final
    .sort_values(by=["Kecamatan", "Periode"])
    .reset_index(drop=True)
)


# ================================================================
# 13. VALIDASI DATA FINAL
# ================================================================

jumlah_kecamatan_final = df_final["Kecamatan"].nunique()
jumlah_periode_final = df_final["Periode"].nunique()
jumlah_baris_seharusnya = len(DAFTAR_KECAMATAN) * len(daftar_periode)

print("\n" + "=" * 75)
print("VALIDASI DATA FINAL")
print("=" * 75)
print("Jumlah kecamatan:", jumlah_kecamatan_final)
print("Jumlah periode:", jumlah_periode_final)
print("Periode awal:", df_final["Periode"].min())
print("Periode akhir:", df_final["Periode"].max())
print("Jumlah baris aktual:", len(df_final))
print("Jumlah baris seharusnya:", jumlah_baris_seharusnya)
print("Total kasus historis 2022–2024:", df_historis["Jumlah_Kasus"].sum())
print("Total kasus 2025:", df_2025_m["Jumlah_Kasus"].sum())
print("Total kasus keseluruhan:", df_final["Jumlah_Kasus"].sum())

assert jumlah_kecamatan_final == 27, f"Jumlah kecamatan adalah {jumlah_kecamatan_final}, seharusnya 27."
assert len(df_final) == jumlah_baris_seharusnya, "Jumlah baris panel tidak sesuai."
assert (df_final["Jumlah_Kasus"] >= 0).all(), "Ditemukan jumlah kasus negatif."
assert (df_final.duplicated(subset=["Kecamatan", "Tahun", "Bulan"]).sum() == 0), "Ditemukan duplikasi kecamatan-tahun-bulan."
assert (df_final["Jumlah_Kasus"].sum() == df_master["Jumlah_Kasus"].sum()), "Total kasus berubah setelah pembentukan panel."

print("\n[SUKSES] Data bulanan 2022–2025 berhasil diselaraskan.")
print("[SUKSES] Jumlah kecamatan tepat 27.")
print("[SUKSES] Tidak ada periode masa depan yang dipaksakan menjadi nol.")
display(df_final.head(15))


if "df_final" not in globals():
    kandidat_checkpoint = [
        os.path.join(path_folder, "Dataset_Campak_Sumenep_2022_2025_Long_Format.csv"),
        os.path.join(path_folder, "df_final_bulanan.csv")
    ]
    checkpoint_ditemukan = next((p for p in kandidat_checkpoint if os.path.exists(p)), None)

    if checkpoint_ditemukan is None:
        raise FileNotFoundError(
            "Variabel df_final belum tersedia dan checkpoint CSV tidak ditemukan. "
            "Jalankan terlebih dahulu cell penyelarasan data yang sudah diperbaiki."
        )

    df_final = pd.read_csv(checkpoint_ditemukan)
    print("[INFO] df_final dimuat dari:", checkpoint_ditemukan)

kolom_wajib = ["Kecamatan", "Tahun", "Bulan", "Jumlah_Kasus"]
kolom_hilang = [kolom for kolom in kolom_wajib if kolom not in df_final.columns]

if kolom_hilang:
    raise ValueError(f"Kolom wajib belum tersedia: {kolom_hilang}")

df_final = df_final.copy()

df_final["Kecamatan"] = (
    df_final["Kecamatan"]
    .astype(str)
    .str.strip()
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
)

df_final["Tahun"] = pd.to_numeric(df_final["Tahun"], errors="coerce")
df_final["Bulan"] = pd.to_numeric(df_final["Bulan"], errors="coerce")
df_final["Jumlah_Kasus"] = pd.to_numeric(df_final["Jumlah_Kasus"], errors="coerce")

df_final = df_final.dropna(subset=["Kecamatan", "Tahun", "Bulan", "Jumlah_Kasus"]).copy()

df_final["Tahun"] = df_final["Tahun"].astype(int)
df_final["Bulan"] = df_final["Bulan"].astype(int)
df_final["Jumlah_Kasus"] = df_final["Jumlah_Kasus"].astype(float)

if not df_final["Bulan"].between(1, 12).all():
    raise ValueError("Ditemukan nilai bulan di luar rentang 1–12.")

if (df_final["Jumlah_Kasus"] < 0).any():
    raise ValueError("Ditemukan jumlah kasus negatif.")

df_final["Periode"] = pd.to_datetime(
    dict(
        year=df_final["Tahun"],
        month=df_final["Bulan"],
        day=1
    )
)

df_final = (
    df_final
    .groupby(["Kecamatan", "Tahun", "Bulan", "Periode"], as_index=False)["Jumlah_Kasus"]
    .sum()
)

df_final = (
    df_final
    .sort_values(["Kecamatan", "Periode"])
    .reset_index(drop=True)
)

jumlah_kecamatan = df_final["Kecamatan"].nunique()
jumlah_periode = df_final["Periode"].nunique()
jumlah_baris_seharusnya = jumlah_kecamatan * jumlah_periode

print("=" * 72)
print("VALIDASI DATASET PANEL")
print("=" * 72)
print("Jumlah kecamatan :", jumlah_kecamatan)
print("Jumlah periode   :", jumlah_periode)
print("Periode awal     :", df_final["Periode"].min().strftime("%Y-%m"))
print("Periode akhir    :", df_final["Periode"].max().strftime("%Y-%m"))
print("Jumlah baris     :", len(df_final))
print("Baris seharusnya :", jumlah_baris_seharusnya)
print("Total kasus      :", int(df_final["Jumlah_Kasus"].sum()))

if jumlah_kecamatan != 27:
    raise ValueError(
        f"Jumlah kecamatan masih {jumlah_kecamatan}. "
        "Standarisasi nama wilayah harus diperbaiki hingga tepat 27 kecamatan."
    )

if len(df_final) != jumlah_baris_seharusnya:
    raise ValueError(
        "Dataset belum membentuk balanced panel. "
        "Periksa kecamatan-periode yang hilang sebelum pemodelan."
    )

display(df_final.head(10))

# ----------------------------------------------------------------
# DETEKSI POLA DISAGREGASI TAHUNAN SECARA RATA
# ----------------------------------------------------------------

def cocok_dengan_pembagian_rata(grup):
    grup = grup.sort_values("Bulan")
    if len(grup) != 12:
        return False

    nilai = grup["Jumlah_Kasus"].round().astype(int).to_numpy()
    total = int(nilai.sum())
    nilai_dasar, sisa = divmod(total, 12)

    ekspektasi = np.array([
        nilai_dasar + (1 if bulan <= sisa else 0)
        for bulan in range(1, 13)
    ])

    return bool(np.array_equal(nilai, ekspektasi))

data_historis_deteksi = df_final[df_final["Tahun"] < TAHUN_TEST].copy()

indikasi_disagregasi = (
    data_historis_deteksi
    .groupby(["Kecamatan", "Tahun"])
    .apply(cocok_dengan_pembagian_rata)
)

jumlah_terindikasi = int(indikasi_disagregasi.sum())
jumlah_grup = int(len(indikasi_disagregasi))
persentase_terindikasi = (100 * jumlah_terindikasi / jumlah_grup if jumlah_grup > 0 else 0)

print("=" * 72)
print("PEMERIKSAAN POLA BULANAN BUATAN")
print("=" * 72)
print("Kelompok kecamatan-tahun diperiksa :", jumlah_grup)
print("Terindikasi pembagian rata         :", jumlah_terindikasi)
print(f"Persentase indikasi                : {persentase_terindikasi:.2f}%")

if persentase_terindikasi >= 50:
    pesan = (
        "Mayoritas data historis cocok dengan pola pembagian total tahunan "
        "secara rata. Hasil lag, feature importance, dan evaluasi model akan "
        "bias. Gunakan data bulanan observasi sebelum melanjutkan."
    )
    if HENTIKAN_JIKA_DISAGREGASI_RATA:
        raise ValueError(pesan)
    else:
        print("[PERINGATAN KERAS]", pesan)
else:
    print("[LOLOS] Tidak ditemukan indikasi dominan pembagian rata tahunan.")

# Tahap 3 Data Understanding dan Analisis Temporal

ringkasan_kualitas = pd.DataFrame({
    "Pemeriksaan": [
        "Jumlah baris",
        "Jumlah kecamatan",
        "Jumlah periode",
        "Missing Kecamatan",
        "Missing Jumlah Kasus",
        "Duplikasi kecamatan-periode",
        "Nilai kasus negatif",
        "Observasi bernilai nol"
    ],
    "Nilai": [
        len(df_final),
        df_final["Kecamatan"].nunique(),
        df_final["Periode"].nunique(),
        int(df_final["Kecamatan"].isna().sum()),
        int(df_final["Jumlah_Kasus"].isna().sum()),
        int(df_final.duplicated(["Kecamatan", "Periode"]).sum()),
        int((df_final["Jumlah_Kasus"] < 0).sum()),
        int((df_final["Jumlah_Kasus"] == 0).sum())
    ]
})

rekap_tahunan = (
    df_final
    .groupby("Tahun", as_index=False)
    .agg(
        Jumlah_Observasi=("Jumlah_Kasus", "size"),
        Total_Kasus=("Jumlah_Kasus", "sum"),
        Rata_Rata_Bulanan_Kecamatan=("Jumlah_Kasus", "mean"),
        Median=("Jumlah_Kasus", "median"),
        Maksimum=("Jumlah_Kasus", "max")
    )
)

rekap_bulanan = (
    df_final
    .groupby("Periode", as_index=False)["Jumlah_Kasus"]
    .sum()
    .rename(columns={"Jumlah_Kasus": "Total_Kasus"})
)

print("TABEL PEMERIKSAAN KUALITAS DATA")
display(ringkasan_kualitas)

print("TABEL REKAPITULASI TAHUNAN")
display(rekap_tahunan.round(3))

plt.figure(figsize=(11, 5))
plt.plot(rekap_bulanan["Periode"], rekap_bulanan["Total_Kasus"], marker="o")
plt.title("Tren Total Kasus Campak Bulanan")
plt.xlabel("Periode")
plt.ylabel("Total kasus seluruh kecamatan")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(path_output, "01_tren_kasus_bulanan.png"), dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(rekap_tahunan["Tahun"], rekap_tahunan["Total_Kasus"], marker="o")
plt.title("Tren Total Kasus Campak Tahunan")
plt.xlabel("Tahun")
plt.ylabel("Total kasus")
plt.xticks(rekap_tahunan["Tahun"])
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(path_output, "02_tren_kasus_tahunan.png"), dpi=300, bbox_inches="tight")
plt.show()

# Tahap 4 Persiapan Data Spasial

def standardisasi_kecamatan_spasial(series):
    mapping = {
        "AMBUTEN": "AMBUNTEN",
        "ARJASA PULAU KANGEAN": "ARJASA",
        "ARJASA (PULAU KANGEAN)": "ARJASA",
        "BATANG-BATANG": "BATANG BATANG",
        "BATANGBATANG": "BATANG BATANG",
        "BATU PUTIH": "BATUPUTIH",
        "GILI GENTENG": "GILIGENTING",
        "GILI GENTING": "GILIGENTING",
        "GILIGENTENG": "GILIGENTING",
        "GILI-GENTENG": "GILIGENTING",
        "GILI-GENTING": "GILIGENTING",
        "GILIGINTING": "GILIGENTING",
        "GULUK-GULUK": "GULUK GULUK",
        "GULUKGULUK": "GULUK GULUK",
        "KECAMATAN KOTA SUMENEP": "KOTA SUMENEP",
        "SUMENEP": "KOTA SUMENEP",
        "NONG GUNONG": "NONGGUNONG",
        "NONG-GUNONG": "NONGGUNONG",
        "RA'AS": "RAAS"
    }

    hasil = (
        series.astype(str)
        .str.strip()
        .str.upper()
        .str.replace(r"^KEC(?:AMATAN)?\.?\s+", "", regex=True)
        .str.replace(r"[()]", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace(mapping)
    )
    return hasil

df_final["Kecamatan"] = standardisasi_kecamatan_spasial(df_final["Kecamatan"])

nama_file_peta = "batas_kecamatan_sumenep.geojson"
path_peta = os.path.join(path_folder, nama_file_peta)

if not os.path.exists(path_peta):
    raise FileNotFoundError(f"File peta tidak ditemukan: {path_peta}")

gdf_sumenep = gpd.read_file(path_peta)

kandidat_kolom = ["Kecamatan", "KECAMATAN", "kecamatan", "NAME_2", "NAME_3", "KEC", "kec"]
kolom_kecamatan_peta = next((kolom for kolom in kandidat_kolom if kolom in gdf_sumenep.columns), None)

if kolom_kecamatan_peta is None:
    kandidat_objek = [
        kolom for kolom in gdf_sumenep.columns
        if kolom != "geometry" and gdf_sumenep[kolom].dtype == "object"
    ]
    if not kandidat_objek:
        raise ValueError("Kolom nama kecamatan pada GeoJSON tidak ditemukan.")
    kolom_kecamatan_peta = kandidat_objek[0]

gdf_sumenep["Kecamatan_Clean"] = standardisasi_kecamatan_spasial(gdf_sumenep[kolom_kecamatan_peta])

gdf_sumenep = (
    gdf_sumenep[["Kecamatan_Clean", "geometry"]]
    .dissolve(by="Kecamatan_Clean", as_index=False)
    .reset_index(drop=True)
)

if not gdf_sumenep.geometry.is_valid.all():
    gdf_sumenep["geometry"] = gdf_sumenep.geometry.buffer(0)

data_spasial_2025 = (
    df_final[df_final["Tahun"] == TAHUN_TEST]
    .groupby("Kecamatan", as_index=False)["Jumlah_Kasus"]
    .sum()
)

gdf_spasial = gdf_sumenep.merge(
    data_spasial_2025,
    left_on="Kecamatan_Clean",
    right_on="Kecamatan",
    how="left",
    validate="one_to_one"
)

kecamatan_tidak_match = gdf_spasial.loc[gdf_spasial["Kecamatan"].isna(), "Kecamatan_Clean"].tolist()

if kecamatan_tidak_match:
    raise ValueError(f"Nama kecamatan pada data dan GeoJSON belum cocok: {kecamatan_tidak_match}")

gdf_spasial["Jumlah_Kasus"] = gdf_spasial["Jumlah_Kasus"].fillna(0).astype(float)
gdf_spasial = gdf_spasial.reset_index(drop=True)

print("Kolom nama peta    :", kolom_kecamatan_peta)
print("Jumlah wilayah peta:", len(gdf_spasial))
print("CRS peta           :", gdf_spasial.crs)
display(gdf_spasial[["Kecamatan_Clean", "Jumlah_Kasus"]].head())


# TAHAP 5 Uji Sensitivitas KNN dan Perbandingan Queen

crs_metrik = gdf_spasial.estimate_utm_crs()

if crs_metrik is None:
    raise ValueError("CRS metrik tidak dapat diestimasi dari geometri.")

gdf_metrik = gdf_spasial.to_crs(crs_metrik).reset_index(drop=True)
nilai_spasial = gdf_spasial["Jumlah_Kasus"].to_numpy(dtype=float)

if np.isclose(np.var(nilai_spasial), 0):
    raise ValueError("Varians jumlah kasus adalah nol. Moran's I tidak dapat dihitung.")

hasil_sensitivitas_k = []

for k in range(3, 8):
    w_k = KNN.from_dataframe(gdf_metrik, k=k, use_index=True)
    w_k.transform = "R"
    moran_k = Moran(nilai_spasial, w_k, permutations=999)

    hasil_sensitivitas_k.append({
        "Nilai_k": k,
        "Moran_I": moran_k.I,
        "Expected_I": moran_k.EI,
        "Z_Score_Permutasi": moran_k.z_sim,
        "P_Value_Permutasi": moran_k.p_sim,
        "Jumlah_Komponen": w_k.n_components,
        "Jumlah_Island": len(w_k.islands)
    })

df_sensitivitas_k = pd.DataFrame(hasil_sensitivitas_k)
print("TABEL UJI SENSITIVITAS KNN k=3 SAMPAI k=7")
display(df_sensitivitas_k.round(4))

plt.figure(figsize=(8, 5))
plt.plot(df_sensitivitas_k["Nilai_k"], df_sensitivitas_k["Moran_I"], marker="o", label="Moran's I")
plt.axhline(df_sensitivitas_k["Expected_I"].iloc[0], linestyle="--", label="Expected Moran's I")
plt.axvline(K_UTAMA, linestyle=":", label=f"K utama = {K_UTAMA}")
plt.title("Sensitivitas Moran's I terhadap Nilai k")
plt.xlabel("Nilai k")
plt.ylabel("Moran's I")
plt.xticks(df_sensitivitas_k["Nilai_k"])
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(path_output, "03_sensitivitas_knn.png"), dpi=300, bbox_inches="tight")
plt.show()

w_queen = Queen.from_dataframe(gdf_metrik, use_index=True)
w_queen.transform = "R"

w_knn = KNN.from_dataframe(gdf_metrik, k=K_UTAMA, use_index=True)
w_knn.transform = "R"

moran_queen = Moran(nilai_spasial, w_queen, permutations=999)
moran_knn = Moran(nilai_spasial, w_knn, permutations=999)

def interpretasi_moran(objek_moran):
    if objek_moran.p_sim >= 0.05:
        return "Tidak signifikan / pola tidak berbeda dari acak"
    if objek_moran.I > objek_moran.EI:
        return "Autokorelasi positif / mengelompok"
    return "Autokorelasi negatif / menyebar"

df_perbandingan_spasial = pd.DataFrame({
    "Metode": ["Queen Contiguity", f"KNN (k={K_UTAMA})"],
    "Moran_I": [moran_queen.I, moran_knn.I],
    "Expected_Moran_I": [moran_queen.EI, moran_knn.EI],
    "Z_Score_Permutasi": [moran_queen.z_sim, moran_knn.z_sim],
    "P_Value_Permutasi": [moran_queen.p_sim, moran_knn.p_sim],
    "Jumlah_Komponen": [w_queen.n_components, w_knn.n_components],
    "Jumlah_Island": [len(w_queen.islands), len(w_knn.islands)],
    "Kesimpulan": [interpretasi_moran(moran_queen), interpretasi_moran(moran_knn)]
})

print("TABEL PERBANDINGAN GLOBAL MORAN'S I")
display(df_perbandingan_spasial.round(4))
print("Queen islands:", w_queen.islands)
print("KNN islands  :", w_knn.islands)

# TAHAP 6 Moran Scatterplot

def plot_moran_scatter(nilai, bobot, judul, nama_file):
    z = (nilai - np.mean(nilai)) / np.std(nilai, ddof=0)
    spatial_lag_z = lag_spatial(bobot, z)
    koefisien = np.polyfit(z, spatial_lag_z, 1)
    garis = np.poly1d(koefisien)

    plt.figure(figsize=(7, 6))
    plt.scatter(z, spatial_lag_z, alpha=0.8)
    plt.plot(np.sort(z), garis(np.sort(z)), linestyle="--", label=f"Slope = {koefisien[0]:.4f}")
    plt.axhline(0, linewidth=1)
    plt.axvline(0, linewidth=1)
    plt.title(judul)
    plt.xlabel("Nilai kasus terstandarisasi (z)")
    plt.ylabel("Spatial lag terstandarisasi")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(path_output, nama_file), dpi=300, bbox_inches="tight")
    plt.show()

plot_moran_scatter(nilai_spasial, w_queen, "Moran Scatterplot — Queen Contiguity", "04_moran_scatter_queen.png")
plot_moran_scatter(nilai_spasial, w_knn, f"Moran Scatterplot — KNN (k={K_UTAMA})", "05_moran_scatter_knn.png")

# LISA

lisa_queen = Moran_Local(nilai_spasial, w_queen, permutations=999, seed=RANDOM_STATE)
lisa_knn = Moran_Local(nilai_spasial, w_knn, permutations=999, seed=RANDOM_STATE)

mapping_kuadran = {
    1: "High-High (HH)",
    2: "Low-High (LH)",
    3: "Low-Low (LL)",
    4: "High-Low (HL)"
}

def kategori_lisa(local_moran, alpha=0.05):
    hasil = []
    for q, p in zip(local_moran.q, local_moran.p_sim):
        if p < alpha:
            hasil.append(mapping_kuadran[int(q)])
        else:
            hasil.append("Tidak Signifikan")
    return hasil

gdf_lisa = gdf_spasial.copy()
gdf_lisa["Local_I_Queen"] = lisa_queen.Is
gdf_lisa["P_LISA_Queen"] = lisa_queen.p_sim
gdf_lisa["Kategori_Queen"] = kategori_lisa(lisa_queen)

gdf_lisa["Local_I_KNN"] = lisa_knn.Is
gdf_lisa["P_LISA_KNN"] = lisa_knn.p_sim
gdf_lisa["Kategori_KNN"] = kategori_lisa(lisa_knn)

df_lisa_hasil = gdf_lisa[
    ["Kecamatan_Clean", "Jumlah_Kasus", "Local_I_Queen", "P_LISA_Queen", "Kategori_Queen", "Local_I_KNN", "P_LISA_KNN", "Kategori_KNN"]
].copy()

print("TABEL HASIL LOCAL MORAN'S I / LISA")
display(df_lisa_hasil.sort_values("Kecamatan_Clean").round(4))

warna_lisa = {
    "High-High (HH)": "#d73027",
    "Low-Low (LL)": "#4575b4",
    "High-Low (HL)": "#fdae61",
    "Low-High (LH)": "#74add1",
    "Tidak Signifikan": "#d9d9d9"
}

fig, axes = plt.subplots(1, 2, figsize=(20, 9))

for kategori, warna in warna_lisa.items():
    subset_queen = gdf_lisa[gdf_lisa["Kategori_Queen"] == kategori]
    subset_knn = gdf_lisa[gdf_lisa["Kategori_KNN"] == kategori]

    if not subset_queen.empty:
        subset_queen.plot(ax=axes[0], color=warna, edgecolor="black", linewidth=0.5, label=kategori)
    if not subset_knn.empty:
        subset_knn.plot(ax=axes[1], color=warna, edgecolor="black", linewidth=0.5, label=kategori)

axes[0].set_title("Klaster LISA — Queen Contiguity")
axes[1].set_title(f"Klaster LISA — KNN (k={K_UTAMA})")

for ax in axes:
    ax.set_axis_off()
    ax.legend(loc="lower left", fontsize=8)

plt.suptitle("Perbandingan Klaster Lokal Kasus Campak Tahun 2025", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(path_output, "06_perbandingan_lisa.png"), dpi=300, bbox_inches="tight")
plt.show()

# TABEL RINGKASAN HASIL LISA QUEEN DAN KNN k=5

kolom_wajib = ["Kecamatan_Clean", "Kategori_Queen", "Kategori_KNN"]
kolom_tidak_tersedia = [kolom for kolom in kolom_wajib if kolom not in gdf_lisa.columns]

if kolom_tidak_tersedia:
    raise ValueError(f"Kolom belum tersedia: {kolom_tidak_tersedia}. Jalankan terlebih dahulu perhitungan LISA.")

urutan_kategori = ["High-High (HH)", "Low-Low (LL)", "High-Low (HL)", "Low-High (LH)", "Tidak Signifikan"]

def buat_ringkasan_lisa(data, kolom_kategori, nama_metode):
    hasil_ringkasan = []
    for kategori in urutan_kategori:
        daftar_kecamatan = (
            data.loc[data[kolom_kategori] == kategori, "Kecamatan_Clean"]
            .dropna().astype(str).sort_values().unique().tolist()
        )
        jumlah_kecamatan = len(daftar_kecamatan)
        daftar_teks = ", ".join(daftar_kecamatan) if jumlah_kecamatan > 0 else "–"

        hasil_ringkasan.append({
            "Metode": nama_metode,
            "Kategori": kategori,
            "Jumlah Kecamatan": jumlah_kecamatan,
            "Daftar Kecamatan": daftar_teks
        })
    return pd.DataFrame(hasil_ringkasan)

ringkasan_lisa_queen = buat_ringkasan_lisa(data=gdf_lisa, kolom_kategori="Kategori_Queen", nama_metode="Queen Contiguity")
ringkasan_lisa_knn = buat_ringkasan_lisa(data=gdf_lisa, kolom_kategori="Kategori_KNN", nama_metode=f"KNN (k={K_UTAMA})")

print("=" * 90)
print("RINGKASAN HASIL LISA QUEEN CONTIGUITY")
print("=" * 90)
display(ringkasan_lisa_queen[["Kategori", "Jumlah Kecamatan", "Daftar Kecamatan"]])

print("\n" + "=" * 90)
print(f"RINGKASAN HASIL LISA KNN k={K_UTAMA}")
print("=" * 90)
display(ringkasan_lisa_knn[["Kategori", "Jumlah Kecamatan", "Daftar Kecamatan"]])

tabel_perbandingan_lisa = pd.DataFrame({"Kategori": urutan_kategori})

tabel_perbandingan_lisa = (
    tabel_perbandingan_lisa
    .merge(
        ringkasan_lisa_queen[["Kategori", "Jumlah Kecamatan", "Daftar Kecamatan"]]
        .rename(columns={"Jumlah Kecamatan": "Jumlah Queen", "Daftar Kecamatan": "Daftar Kecamatan Queen"}),
        on="Kategori", how="left"
    )
    .merge(
        ringkasan_lisa_knn[["Kategori", "Jumlah Kecamatan", "Daftar Kecamatan"]]
        .rename(columns={"Jumlah Kecamatan": "Jumlah KNN", "Daftar Kecamatan": "Daftar Kecamatan KNN"}),
        on="Kategori", how="left"
    )
)

print("\n" + "=" * 100)
print("TABEL PERBANDINGAN RINGKASAN HASIL LISA")
print("=" * 100)
display(tabel_perbandingan_lisa)


# TAHAP 7 Daftar Tetangga KNN

centroid_metrik = gdf_metrik.geometry.centroid
baris_tetangga = []

for idx_asal, indeks_tetangga in w_knn.neighbors.items():
    nama_asal = gdf_spasial.loc[idx_asal, "Kecamatan_Clean"]
    record = {"Kecamatan_Acuan": nama_asal}

    for urutan, idx_tetangga in enumerate(indeks_tetangga, start=1):
        nama_tetangga = gdf_spasial.loc[idx_tetangga, "Kecamatan_Clean"]
        jarak_km = centroid_metrik.iloc[idx_asal].distance(centroid_metrik.iloc[idx_tetangga]) / 1000
        record[f"Tetangga_{urutan}"] = nama_tetangga
        record[f"Jarak_{urutan}_km"] = jarak_km

    baris_tetangga.append(record)

df_tetangga_knn = pd.DataFrame(baris_tetangga).sort_values("Kecamatan_Acuan").reset_index(drop=True)

print(f"TABEL MATRIKS TETANGGA KNN k={K_UTAMA}")
display(df_tetangga_knn.round(3))

# ================================================================
# TABEL 4.8 RINGKASAN KLASTER LISA KNN k=5
# ================================================================

kolom_wajib_lisa = ["Kecamatan_Clean", "Kategori_KNN"]
kolom_hilang = [kolom for kolom in kolom_wajib_lisa if kolom not in gdf_lisa.columns]

if kolom_hilang:
    raise ValueError(f"Kolom berikut belum tersedia: {kolom_hilang}. Jalankan terlebih dahulu cell perhitungan LISA KNN.")

urutan_kategori = ["High-High (HH)", "Low-Low (LL)", "High-Low (HL)", "Low-High (LH)", "Tidak Signifikan"]

nama_kategori_tabel = {
    "High-High (HH)": "High-High (HH)",
    "Low-Low (LL)": "Low-Low (LL)",
    "High-Low (HL)": "High-Low (HL)",
    "Low-High (LH)": "Low-High (LH)",
    "Tidak Signifikan": "Tidak signifikan"
}

ringkasan_lisa_knn = []

for kategori in urutan_kategori:
    daftar_kecamatan = (
        gdf_lisa.loc[gdf_lisa["Kategori_KNN"] == kategori, "Kecamatan_Clean"]
        .dropna().astype(str).sort_values().unique().tolist()
    )
    jumlah_kecamatan = len(daftar_kecamatan)
    daftar_teks = "-" if jumlah_kecamatan == 0 else ", ".join(daftar_kecamatan)

    ringkasan_lisa_knn.append({
        "Kategori": nama_kategori_tabel[kategori],
        "Jumlah Kecamatan": jumlah_kecamatan,
        "Daftar Kecamatan": daftar_teks
    })

tabel_4_8_lisa_knn = pd.DataFrame(ringkasan_lisa_knn)

total_kecamatan_lisa = tabel_4_8_lisa_knn["Jumlah Kecamatan"].sum()
jumlah_kecamatan_unik = gdf_lisa["Kecamatan_Clean"].nunique()

print("=" * 75)
print("TABEL 4.8 RINGKASAN KLASTER LISA KNN k=5")
print("=" * 75)
display(tabel_4_8_lisa_knn)

assert total_kecamatan_lisa == jumlah_kecamatan_unik, "Total kecamatan pada kategori LISA tidak sesuai dengan jumlah kecamatan pada data."
assert jumlah_kecamatan_unik == 27, f"Jumlah kecamatan adalah {jumlah_kecamatan_unik}, seharusnya 27 kecamatan."

print("[SUKSES] Seluruh 27 kecamatan telah masuk ke dalam ringkasan kategori LISA KNN.")

# TAHAP 8 Pembentukan Lag Feature

df_model = df_final.sort_values(["Kecamatan", "Periode"]).reset_index(drop=True).copy()
lag_kolom = []

for lag in range(1, JUMLAH_LAG + 1):
    nama_lag = f"Lag_{lag}"
    lag_kolom.append(nama_lag)
    df_model[nama_lag] = df_model.groupby("Kecamatan")["Jumlah_Kasus"].shift(lag)

df_model["Bulan"] = df_model["Periode"].dt.month
df_model["Tahun"] = df_model["Periode"].dt.year

contoh_kecamatan = df_model["Kecamatan"].sort_values().iloc[0]
tabel_simulasi_lag = df_model[df_model["Kecamatan"] == contoh_kecamatan][["Kecamatan", "Periode", "Jumlah_Kasus"] + lag_kolom].head(10)

print("TABEL SIMULASI PEMBENTUKAN LAG FEATURE")
display(tabel_simulasi_lag)

df_model_clean = (
    df_model.dropna(subset=lag_kolom + ["Jumlah_Kasus"])
    .sort_values(["Periode", "Kecamatan"])
    .reset_index(drop=True)
)

fitur_kolom = lag_kolom + ["Bulan", "Tahun"]
target_kolom = "Jumlah_Kasus"

print("Fitur model      :", fitur_kolom)
print("Target model     :", target_kolom)
print("Jumlah observasi :", len(df_model_clean))

# TAHAP 9 Time-Series Chronological Split

batas_test = pd.Timestamp(year=TAHUN_TEST, month=1, day=1)

train_data = df_model_clean[df_model_clean["Periode"] < batas_test].sort_values(["Periode", "Kecamatan"]).reset_index(drop=True)
test_data = df_model_clean[df_model_clean["Periode"] >= batas_test].sort_values(["Periode", "Kecamatan"]).reset_index(drop=True)

if train_data.empty or test_data.empty:
    raise ValueError("Data training atau testing kosong.")

X_train = train_data[fitur_kolom].copy()
y_train = train_data[target_kolom].copy()
X_test = test_data[fitur_kolom].copy()
y_test = test_data[target_kolom].copy()

print("=" * 72)
print("TIME-SERIES CHRONOLOGICAL HOLDOUT")
print("=" * 72)
print("Training:", train_data["Periode"].min().strftime("%Y-%m"), "sampai", train_data["Periode"].max().strftime("%Y-%m"))
print("Testing :", test_data["Periode"].min().strftime("%Y-%m"), "sampai", test_data["Periode"].max().strftime("%Y-%m"))

# TAHAP 10 Fungsi Evaluasi dan Model Awal

def evaluasi_regresi(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred)
    }

rf_awal = RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
rf_awal.fit(X_train, y_train)
y_pred_awal = np.maximum(rf_awal.predict(X_test), 0)

y_pred_baseline = X_test["Lag_1"].to_numpy(dtype=float)

metrik_awal = evaluasi_regresi(y_test, y_pred_awal)
metrik_baseline = evaluasi_regresi(y_test, y_pred_baseline)

df_evaluasi_awal = pd.DataFrame([
    {"Model": "Random Forest Awal", **metrik_awal},
    {"Model": "Baseline Lag-1", **metrik_baseline}
])

print("EVALUASI MODEL AWAL DAN BASELINE")
display(df_evaluasi_awal.round(4))

# TAHAP 11 5-Fold Time-Series Cross-Validation

def membuat_split_berdasarkan_periode(data_training, n_splits=5):
    periode_unik = np.array(sorted(data_training["Periode"].unique()))
    tscv = TimeSeriesSplit(n_splits=n_splits)
    daftar_split = []
    ringkasan_fold = []

    for nomor_fold, (idx_periode_train, idx_periode_validasi) in enumerate(tscv.split(periode_unik), start=1):
        periode_fold_train = periode_unik[idx_periode_train]
        periode_fold_validasi = periode_unik[idx_periode_validasi]

        idx_baris_train = np.flatnonzero(data_training["Periode"].isin(periode_fold_train).to_numpy())
        idx_baris_validasi = np.flatnonzero(data_training["Periode"].isin(periode_fold_validasi).to_numpy())

        daftar_split.append((idx_baris_train, idx_baris_validasi))
        ringkasan_fold.append({
            "Fold": nomor_fold,
            "Train_Awal": pd.Timestamp(periode_fold_train[0]),
            "Train_Akhir": pd.Timestamp(periode_fold_train[-1]),
            "Validasi_Awal": pd.Timestamp(periode_fold_validasi[0]),
            "Validasi_Akhir": pd.Timestamp(periode_fold_validasi[-1]),
            "Baris_Train": len(idx_baris_train),
            "Baris_Validasi": len(idx_baris_validasi)
        })
    return daftar_split, pd.DataFrame(ringkasan_fold)

cv_time_series, df_fold_cv = membuat_split_berdasarkan_periode(train_data, n_splits=JUMLAH_FOLD)

print("PEMBAGIAN 5-FOLD TIME-SERIES CROSS-VALIDATION")
display(df_fold_cv)

# TAHAP 12 Hyperparameter Tuning Random Forest

distribusi_parameter = {
    "n_estimators": [200, 300, 500, 700],
    "max_depth": [None, 5, 10, 15, 20],
    "min_samples_split": [2, 5, 10, 15],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": ["sqrt", 0.5, 1.0],
    "bootstrap": [True, False]
}

rf_tuning = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)

random_search = RandomizedSearchCV(
    estimator=rf_tuning,
    param_distributions=distribusi_parameter,
    n_iter=40,
    scoring="neg_root_mean_squared_error",
    cv=cv_time_series,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=2,
    refit=True,
    return_train_score=True
)

print("[INFO] Memulai RandomizedSearchCV berbasis urutan waktu...")
random_search.fit(X_train, y_train)

rf_regressor_tuned = random_search.best_estimator_
y_pred_tuned = np.maximum(rf_regressor_tuned.predict(X_test), 0)
metrik_tuned = evaluasi_regresi(y_test, y_pred_tuned)

indeks_terbaik = random_search.best_index_
cv_rmse_scores = np.array([
    -random_search.cv_results_[f"split{fold}_test_score"][indeks_terbaik]
    for fold in range(len(cv_time_series))
])

print("=" * 72)
print("PARAMETER TERBAIK")
print("=" * 72)
for nama, nilai in random_search.best_params_.items():
    print(f"{nama}: {nilai}")

y_pred = y_pred_tuned

# TAHAP 13 Rekapitulasi Evaluasi Model

df_perbandingan_model = pd.DataFrame([
    {"Model": "Random Forest Awal", **metrik_awal},
    {"Model": "Random Forest Tuned", **metrik_tuned},
    {"Model": "Baseline Lag-1", **metrik_baseline}
])

df_rekap_evaluasi = pd.DataFrame({
    "Metrik_Evaluasi": [
        "MAE Random Forest Tuned",
        "RMSE Random Forest Tuned",
        "R-squared Random Forest Tuned",
        "Rata-rata RMSE 5-Fold Time-Series CV",
        "Standar Deviasi RMSE 5-Fold Time-Series CV",
        "Jumlah aktual nol pada data test"
    ],
    "Nilai": [
        metrik_tuned["MAE"],
        metrik_tuned["RMSE"],
        metrik_tuned["R2"],
        cv_rmse_scores.mean(),
        cv_rmse_scores.std(),
        int((np.asarray(y_test) == 0).sum())
    ]
})

print("PERBANDINGAN MODEL")
display(df_perbandingan_model.round(4))
print("REKAPITULASI METRIK UTAMA")
display(df_rekap_evaluasi.round(4))

# TAHAP 14 Visualisasi Aktual, Prediksi, dan Residual

hasil_prediksi = test_data[["Kecamatan", "Periode", "Tahun", "Bulan", "Jumlah_Kasus"]].copy()
hasil_prediksi = hasil_prediksi.rename(columns={"Jumlah_Kasus": "Aktual"})

hasil_prediksi["Prediksi_RF"] = y_pred_tuned
hasil_prediksi["Prediksi_Baseline"] = y_pred_baseline
hasil_prediksi["Prediksi_RF_Bulat"] = np.rint(y_pred_tuned).clip(min=0).astype(int)
hasil_prediksi["Residual"] = hasil_prediksi["Aktual"] - hasil_prediksi["Prediksi_RF"]
hasil_prediksi["Error_Absolut"] = hasil_prediksi["Residual"].abs()

hasil_periode = hasil_prediksi.groupby("Periode", as_index=False)[["Aktual", "Prediksi_RF", "Prediksi_Baseline"]].sum()

plt.figure(figsize=(12, 5))
plt.plot(hasil_periode["Periode"], hasil_periode["Aktual"], marker="o", label="Aktual")
plt.plot(hasil_periode["Periode"], hasil_periode["Prediksi_RF"], marker="o", label="Random Forest")
plt.plot(hasil_periode["Periode"], hasil_periode["Prediksi_Baseline"], marker="o", linestyle="--", label="Baseline Lag-1")
plt.title("Total Aktual dan Prediksi per Periode Pengujian")
plt.xlabel("Periode")
plt.ylabel("Total kasus seluruh kecamatan")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(path_output, "08_prediksi_per_periode.png"), dpi=300, bbox_inches="tight")
plt.show()

# TAHAP 15 Feature Importance

importance_impurity = pd.DataFrame({
    "Fitur": fitur_kolom,
    "Importance_Impurity": rf_regressor_tuned.feature_importances_
})

hasil_permutation = permutation_importance(
    rf_regressor_tuned, X_test, y_test, scoring="neg_root_mean_squared_error", n_repeats=20, random_state=RANDOM_STATE, n_jobs=-1
)

importance_permutation = pd.DataFrame({
    "Fitur": fitur_kolom,
    "Permutation_Mean": hasil_permutation.importances_mean,
    "Permutation_Std": hasil_permutation.importances_std
})

df_feature_importance = (
    importance_impurity
    .merge(importance_permutation, on="Fitur", how="left")
    .sort_values("Importance_Impurity", ascending=False)
    .reset_index(drop=True)
)

df_feature_importance["Persentase_Impurity"] = df_feature_importance["Importance_Impurity"] * 100

print("TABEL FEATURE IMPORTANCE")
display(df_feature_importance.round(4))

# TAHAP 16 Evaluasi per Periode dan per Kecamatan

def evaluasi_kelompok(grup):
    aktual = grup["Aktual"].to_numpy()
    prediksi = grup["Prediksi_RF"].to_numpy()
    hasil = {
        "MAE": mean_absolute_error(aktual, prediksi),
        "RMSE": np.sqrt(mean_squared_error(aktual, prediksi)),
    }
    if len(grup) >= 2 and not np.isclose(np.var(aktual), 0):
        hasil["R2"] = r2_score(aktual, prediksi)
    else:
        hasil["R2"] = np.nan
    return pd.Series(hasil)

evaluasi_per_periode = hasil_prediksi.groupby("Periode").apply(evaluasi_kelompok).reset_index()
evaluasi_per_kecamatan = hasil_prediksi.groupby("Kecamatan").apply(evaluasi_kelompok).reset_index().sort_values("RMSE", ascending=False)

print("EVALUASI PER PERIODE")
display(evaluasi_per_periode.round(4))

# TAHAP 17 Visualisasi Spasial Aktual dan Prediksi 2025

rekap_spasial_prediksi = (
    hasil_prediksi.groupby("Kecamatan", as_index=False)
    .agg(Total_Aktual=("Aktual", "sum"), Total_Prediksi_RF=("Prediksi_RF", "sum"), MAE_Kecamatan=("Error_Absolut", "mean"))
)

gdf_prediksi_2025 = (
    gdf_spasial[["Kecamatan_Clean", "geometry"]]
    .merge(rekap_spasial_prediksi, left_on="Kecamatan_Clean", right_on="Kecamatan", how="left", validate="one_to_one")
)

# TAHAP 18 Pelatihan Final dan Prediksi Periode Berikutnya

model_deployment = clone(rf_regressor_tuned)
X_seluruh_data = df_model_clean[fitur_kolom].copy()
y_seluruh_data = df_model_clean[target_kolom].copy()

model_deployment.fit(X_seluruh_data, y_seluruh_data)

periode_terakhir = df_final["Periode"].max()
periode_berikutnya = periode_terakhir + pd.offsets.MonthBegin(1)

baris_forecast = []

for kecamatan, grup in df_final.sort_values(["Kecamatan", "Periode"]).groupby("Kecamatan"):
    nilai_terakhir = grup["Jumlah_Kasus"].tail(JUMLAH_LAG).to_numpy(dtype=float)

    if len(nilai_terakhir) < JUMLAH_LAG:
        raise ValueError(f"Riwayat {kecamatan} kurang dari {JUMLAH_LAG} periode.")

    record = {
        "Kecamatan": kecamatan,
        "Periode_Prediksi": periode_berikutnya,
        "Bulan": periode_berikutnya.month,
        "Tahun": periode_berikutnya.year
    }

    for lag in range(1, JUMLAH_LAG + 1):
        record[f"Lag_{lag}"] = nilai_terakhir[-lag]

    baris_forecast.append(record)

df_forecast_berikutnya = pd.DataFrame(baris_forecast).sort_values("Kecamatan").reset_index(drop=True)
X_forecast = df_forecast_berikutnya[fitur_kolom]

df_forecast_berikutnya["Prediksi_Kasus"] = np.maximum(model_deployment.predict(X_forecast), 0)
df_forecast_berikutnya["Prediksi_Kasus_Bulat"] = np.rint(df_forecast_berikutnya["Prediksi_Kasus"]).clip(lower=0).astype(int)

# TAHAP 19 Integrasi Prediksi Berikutnya dengan Analisis Spasial

gdf_forecast = (
    gdf_spasial[["Kecamatan_Clean", "geometry"]]
    .merge(df_forecast_berikutnya[["Kecamatan", "Prediksi_Kasus", "Prediksi_Kasus_Bulat"]], left_on="Kecamatan_Clean", right_on="Kecamatan", how="left")
    .reset_index(drop=True)
)

nilai_prediksi_spasial = gdf_forecast["Prediksi_Kasus"].fillna(0).to_numpy(dtype=float)

if np.isclose(np.var(nilai_prediksi_spasial), 0):
    gdf_forecast["Kategori_LISA_Prediksi"] = "Tidak dapat dihitung"
    print("[PERINGATAN] Varians prediksi nol; Moran dan LISA prediksi tidak dihitung.")
else:
    lisa_prediksi = Moran_Local(nilai_prediksi_spasial, w_knn, permutations=999, seed=RANDOM_STATE)
    gdf_forecast["Kategori_LISA_Prediksi"] = kategori_lisa(lisa_prediksi)

# TAHAP 20 — Tabel Tabel

tabel_4_1 = ringkasan_kualitas.copy()
tabel_4_2 = rekap_tahunan.copy()
tabel_4_3 = df_perbandingan_spasial.copy()
tabel_4_4 = df_tetangga_knn.copy()
tabel_4_5 = df_rekap_evaluasi.copy()
tabel_4_6 = df_feature_importance.copy()
tabel_4_7 = hasil_prediksi.copy()
tabel_4_8 = gdf_forecast.drop(columns="geometry").copy()

# TAHAP 21 Menyimpan Seluruh Output

# CSV utama
df_final.to_csv(os.path.join(path_output, "dataset_final_bulanan.csv"), index=False)
df_sensitivitas_k.to_csv(os.path.join(path_output, "uji_sensitivitas_knn.csv"), index=False)
df_perbandingan_spasial.to_csv(os.path.join(path_output, "perbandingan_moran.csv"), index=False)
df_lisa_hasil.to_csv(os.path.join(path_output, "hasil_lisa.csv"), index=False)
df_tetangga_knn.to_csv(os.path.join(path_output, "tetangga_knn.csv"), index=False)
hasil_prediksi.to_csv(os.path.join(path_output, "hasil_prediksi_test_2025.csv"), index=False)
df_rekap_evaluasi.to_csv(os.path.join(path_output, "rekap_evaluasi_model.csv"), index=False)
df_feature_importance.to_csv(os.path.join(path_output, "feature_importance.csv"), index=False)
evaluasi_per_periode.to_csv(os.path.join(path_output, "evaluasi_per_periode.csv"), index=False)
evaluasi_per_kecamatan.to_csv(os.path.join(path_output, "evaluasi_per_kecamatan.csv"), index=False)
df_forecast_berikutnya.to_csv(os.path.join(path_output, "prediksi_periode_berikutnya.csv"), index=False)

# GeoJSON
gdf_lisa.to_file(os.path.join(path_output, "hasil_lisa.geojson"), driver="GeoJSON")
gdf_prediksi_2025.to_file(os.path.join(path_output, "aktual_prediksi_2025.geojson"), driver="GeoJSON")
gdf_forecast.to_file(os.path.join(path_output, "prediksi_spasial_berikutnya.geojson"), driver="GeoJSON")

# Model
joblib.dump(
    {
        "model": model_deployment,
        "fitur": fitur_kolom,
        "jumlah_lag": JUMLAH_LAG,
        "parameter_terbaik": random_search.best_params_,
        "periode_data_terakhir": str(periode_terakhir.date()),
        "periode_prediksi": str(periode_berikutnya.date())
    },
    os.path.join(path_output, "random_forest_campak.joblib")
)

path_excel = os.path.join(path_output, "Tabel_BAB_IV_Analisis_Campak.xlsx")
with pd.ExcelWriter(path_excel, engine="openpyxl") as writer:
    tabel_4_1.to_excel(writer, sheet_name="Tabel 4.1", index=False)
    tabel_4_2.to_excel(writer, sheet_name="Tabel 4.2", index=False)
    tabel_4_3.to_excel(writer, sheet_name="Tabel 4.3", index=False)
    tabel_4_4.to_excel(writer, sheet_name="Tabel 4.4", index=False)
    tabel_4_5.to_excel(writer, sheet_name="Tabel 4.5", index=False)
    tabel_4_6.to_excel(writer, sheet_name="Tabel 4.6", index=False)
    tabel_4_7.to_excel(writer, sheet_name="Tabel 4.7", index=False)
    tabel_4_8.to_excel(writer, sheet_name="Tabel 4.8", index=False)

import shutil
path_zip_dasar = os.path.join(path_folder, "output_analisis_revisi")
path_zip = shutil.make_archive(path_zip_dasar, "zip", path_output)

print("=" * 72)
print("[SUKSES] SELURUH ANALISIS SELESAI")
print("=" * 72)

# ================================================================
# CATATAN INTERPRETASI AKHIR
# ================================================================
# 1. MAE dan RMSE harus dibaca dalam satuan jumlah kasus.
# 2. R² positif menunjukkan model menjelaskan sebagian variasi, tetapi tidak otomatis berarti akurasi tinggi.
# 3. Random Forest sebaiknya mengungguli baseline Lag-1 agar manfaat model dapat dibuktikan.
# 4. Nilai cross-validation berasal dari expanding-window time-series split, bukan KFold acak.
# 5. LISA menggunakan KNN k=5 sebagai metode utama, sedangkan Queen Contiguity menjadi pembanding.
# 6. Prediksi periode berikutnya merupakan one-step-ahead forecast.
# 7. Kategori prioritas merupakan klasifikasi operasional untuk WebGIS, bukan pengganti uji statistik.
# 8. Jangan menggunakan hasil model sebagai hasil final apabila pemeriksaan mendeteksi data historis hasil pembagian total tahunan secara rata.

# New Section

jumlah_kecamatan = df_final["Kecamatan"].nunique()
jumlah_periode = df_final["Periode"].nunique()
jumlah_baris_seharusnya = jumlah_kecamatan * jumlah_periode

print("=" * 72)
print("VALIDASI DATASET PANEL")
print("=" * 72)

print("Jumlah kecamatan :", jumlah_kecamatan)
print("Jumlah periode   :", jumlah_periode)
print("Periode awal     :", df_final["Periode"].min().strftime("%Y-%m"))
print("Periode akhir    :", df_final["Periode"].max().strftime("%Y-%m"))
print("Jumlah baris     :", len(df_final))
print("Total kasus      :", int(df_final["Jumlah_Kasus"].sum()))

# ================================================================
# RINGKASAN DATASET UNTUK BAB IV
# ================================================================

nama_bulan_indonesia = {
    1: "Januari", 2: "Februari", 3: "Maret", 4: "April",
    5: "Mei", 6: "Juni", 7: "Juli", 8: "Agustus",
    9: "September", 10: "Oktober", 11: "November", 12: "Desember"
}

def format_periode_indonesia(tanggal):
    tanggal = pd.Timestamp(tanggal)
    return f"{nama_bulan_indonesia[tanggal.month]} {tanggal.year}"

jumlah_observasi = len(df_final)
jumlah_kecamatan = df_final["Kecamatan"].nunique()
jumlah_periode = df_final["Periode"].nunique()
periode_awal = format_periode_indonesia(df_final["Periode"].min())
periode_akhir = format_periode_indonesia(df_final["Periode"].max())
total_kasus = int(df_final["Jumlah_Kasus"].sum())

narasi_dataset = (
    f"Setelah proses penyelarasan, dataset akhir terdiri atas "
    f"{jumlah_observasi:,} observasi, "
    f"{jumlah_kecamatan} kecamatan, dan "
    f"{jumlah_periode} periode bulanan dari "
    f"{periode_awal} sampai {periode_akhir}. "
    f"Total kasus yang tercatat pada seluruh periode adalah "
    f"{total_kasus:,} kasus."
)

narasi_dataset = narasi_dataset.replace(",", ".")
print(narasi_dataset)

tabel_ringkasan_dataset = pd.DataFrame({
    "Komponen": [
        "Jumlah kecamatan", "Periode pengamatan", "Jumlah periode bulanan",
        "Jumlah observasi", "Total kasus", "Variabel target",
        "Variabel prediktor", "Data spasial"
    ],
    "Hasil": [
        jumlah_kecamatan, f"{periode_awal}–{periode_akhir}", jumlah_periode,
        jumlah_observasi, total_kasus, "Jumlah_Kasus",
        "Lag_1, Lag_2, Lag_3, Lag_4, Bulan, Tahun", "Batas administrasi kecamatan (GeoJSON)"
    ]
})

display(tabel_ringkasan_dataset)

jumlah_observasi_seharusnya = jumlah_kecamatan * jumlah_periode
print("Jumlah observasi aktual    :", jumlah_observasi)
print("Jumlah observasi seharusnya:", jumlah_observasi_seharusnya)

assert jumlah_observasi == jumlah_observasi_seharusnya, (
    "Dataset belum berbentuk balanced panel. Terdapat kecamatan-periode yang hilang atau duplikat."
)